# 03 Linear SVM: Heart-Disease Prediction and Model Comparison

This notebook applies Linear SVM to the same `heart_data.csv` file.

Linear SVM tries to find a stable separating boundary and keep the two classes as far from that boundary as possible. Like Logistic Regression, it is mostly linear, so it is a useful baseline model.


## Analysis Setup

For a fair comparison, this notebook uses the same setup as the Random Forest notebook:

- Data: `heart_data.csv`
- Target column: `cardio`
- Sample: 12,000 rows sampled after cleaning outliers
- Split: 75% training, 25% testing
- Features: the same feature-engineered inputs
- Metrics: accuracy, precision, recall, F1, and AUC

Note: `LinearSVC` does not output probabilities by default. We use `decision_function` as a ranking score for AUC.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Load Data


In [ ]:
# When this notebook is under /workspace/ml_study/heart, the correct path is ../../data/ml_data/heart_data.csv.
# Try several candidate paths so the notebook runs from different working directories.
candidate_paths = [
    Path('../../data/ml_data/heart_data.csv'),
    Path('../data/ml_data/heart_data.csv'),
    Path('/workspace/data/ml_data/heart_data.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/heart_data.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find heart_data.csv. Please check data/ml_data/heart_data.csv')

heart = pd.read_csv(data_path)
print('Data path:', data_path)
print('Raw data shape:', heart.shape)
heart.head()


## 2. Clean Data and Build Features


In [ ]:
heart = heart.copy()
heart['age_years'] = heart['age'] / 365.25
heart['bmi'] = heart['weight'] / (heart['height'] / 100) ** 2

clean = heart.query(
    '120 <= height <= 220 and 35 <= weight <= 200 and '
    '80 <= ap_hi <= 220 and 40 <= ap_lo <= 140 and 10 <= bmi <= 60'
).copy()

base_features = ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
target = 'cardio'

# Match the Logistic Regression notebook: sample 12,000 rows for speed and fair comparison.
model_data = clean[base_features + [target]].sample(n=12000, random_state=RANDOM_STATE).copy()

# Feature engineering gives the model more useful inputs.
model_data['pulse_pressure'] = model_data['ap_hi'] - model_data['ap_lo']
model_data['mean_arterial_pressure'] = (model_data['ap_hi'] + 2 * model_data['ap_lo']) / 3
model_data['age_bmi'] = model_data['age_years'] * model_data['bmi']
model_data['age_ap_hi'] = model_data['age_years'] * model_data['ap_hi']
model_data['bmi_ap_hi'] = model_data['bmi'] * model_data['ap_hi']
model_data['high_cholesterol'] = (model_data['cholesterol'] >= 2).astype(int)
model_data['very_high_cholesterol'] = (model_data['cholesterol'] >= 3).astype(int)
model_data['high_gluc'] = (model_data['gluc'] >= 2).astype(int)
model_data['stage2_systolic'] = (model_data['ap_hi'] >= 140).astype(int)
model_data['stage2_diastolic'] = (model_data['ap_lo'] >= 90).astype(int)
model_data['older_than_50'] = (model_data['age_years'] >= 50).astype(int)
model_data['older_than_55'] = (model_data['age_years'] >= 55).astype(int)

engineered_features = base_features + [
    'pulse_pressure',
    'mean_arterial_pressure',
    'age_bmi',
    'age_ap_hi',
    'bmi_ap_hi',
    'high_cholesterol',
    'very_high_cholesterol',
    'high_gluc',
    'stage2_systolic',
    'stage2_diastolic',
    'older_than_50',
    'older_than_55',
]

X = model_data[engineered_features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print('Cleaned full data shape:', clean.shape)
print('Modeling data shape:', model_data.shape)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Target distribution in modeling data:')
print(y.value_counts(normalize=True).sort_index().round(3))


## 3. Train Linear SVM

SVM is sensitive to feature scale, so `StandardScaler` is required first.

Here `C=0.03`:

- Smaller `C` makes the boundary more conservative.
- Larger `C` makes the model fit the training set more aggressively.

We also train Logistic Regression and Random Forest as comparison models.


In [ ]:
# This helper formats one model result as a table row for comparison.
def make_metric_row(model_name, y_true, predicted_label, score_for_auc):
    # model_name is the model label, such as Logistic Regression, Random Forest, or Linear SVM.
    return {
        # Store the model name.
        'model': model_name,
        # accuracy is the overall proportion of correct predictions.
        'accuracy': accuracy_score(y_true, predicted_label),
        # precision measures how many predicted cardio=1 samples are truly 1.
        'precision': precision_score(y_true, predicted_label),
        # recall measures how many true cardio=1 samples the model finds.
        'recall': recall_score(y_true, predicted_label),
        # f1 combines precision and recall.
        'f1': f1_score(y_true, predicted_label),
        # roc_auc measures how well the model ranks cardio=1 ahead of cardio=0.
        'roc_auc': roc_auc_score(y_true, score_for_auc),
    }


# This helper rounds metrics to 4 decimals for readability.
def round_metric_table(table):
    # Copy the table to avoid modifying the original object.
    rounded = table.copy()
    # These columns are decimal-valued evaluation metrics.
    for column in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        # Round each metric to 4 decimals.
        rounded[column] = rounded[column].round(4)
    # Return the formatted table.
    return rounded


# Build Logistic Regression as the linear-model comparison.
logistic_model = make_pipeline(
    # Logistic Regression needs scaling so age, blood pressure, and BMI use comparable numeric scales.
    StandardScaler(),
    # C=0.01 is a stable setting from the earlier tuning step.
    LogisticRegression(max_iter=3000, C=0.01, random_state=RANDOM_STATE)
)

# Build a Random Forest as the non-linear comparison model.
random_forest = RandomForestClassifier(
    # n_estimators=160 means the forest has 160 trees.
    n_estimators=160,
    # max_depth=8 controls each tree's complexity.
    max_depth=8,
    # min_samples_leaf=20 requires at least 20 samples per leaf to reduce overfitting.
    min_samples_leaf=20,
    # random_state fixes randomness so results are reproducible.
    random_state=RANDOM_STATE,
    # n_jobs=1 uses one process so the teaching notebook does not use all machine resources.
    n_jobs=1,
)

# Build Linear SVM, which classifies with a stable separating boundary.
linear_svm = make_pipeline(
    # SVM is sensitive to numeric scale, so scaling is required first.
    StandardScaler(),
    # C=0.03 keeps the model conservative and favors a stable boundary.
    LinearSVC(C=0.03, max_iter=10000, random_state=RANDOM_STATE)
)

# Train Logistic Regression on the training set.
logistic_model.fit(X_train, y_train)

# Train Random Forest on the training set.
random_forest.fit(X_train, y_train)

# Train Linear SVM on the training set.
linear_svm.fit(X_train, y_train)

# Get final 0/1 predictions from Logistic Regression on the test set.
logistic_pred = logistic_model.predict(X_test)

# Logistic Regression outputs cardio=1 probability scores for AUC.
logistic_score = logistic_model.predict_proba(X_test)[:, 1]

# Get final 0/1 predictions from Random Forest on the test set.
rf_pred = random_forest.predict(X_test)

# Random Forest also outputs cardio=1 probability scores for AUC.
rf_score = random_forest.predict_proba(X_test)[:, 1]

# Get final 0/1 predictions from Linear SVM on the test set.
svm_pred = linear_svm.predict(X_test)

# LinearSVC does not output probabilities directly; decision_function gives distance scores that can be used for AUC.
svm_score = linear_svm.decision_function(X_test)

# Put the three model results into one comparison table.
comparison = pd.DataFrame([
    # First row: Logistic Regression result.
    make_metric_row('Logistic Regression', y_test, logistic_pred, logistic_score),
    # Second row: Random Forest result.
    make_metric_row('Random Forest', y_test, rf_pred, rf_score),
    # Third row: Linear SVM result.
    make_metric_row('Linear SVM', y_test, svm_pred, svm_score),
])

# Display the comparison table rounded to 4 decimals.
round_metric_table(comparison)


## 4. How to Read the SVM Result

Linear SVM and Logistic Regression are both mostly linear, so their results are usually close.

If Random Forest is slightly better, the data may contain some non-linear relationships. If SVM and Logistic Regression are similar, simply switching to another linear model is unlikely to create a large improvement.


In [ ]:
cm = confusion_matrix(y_test, svm_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['cardio=0', 'cardio=1'])
disp.plot(cmap='Purples')
plt.title('Linear SVM Confusion Matrix')
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, logistic_score, name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, rf_score, name='Random Forest')
RocCurveDisplay.from_predictions(y_test, svm_score, name='Linear SVM')
plt.title('ROC Curve Comparison')
plt.show()


## 5. Inspect Linear SVM Coefficients

Linear SVM gives each feature a coefficient. A positive coefficient leans toward predicting `cardio=1`; a negative coefficient leans toward predicting `cardio=0`.

This is not medical causality. It is only the direction learned by the model from this dataset.


In [ ]:
svm_step = linear_svm.named_steps['linearsvc']
coef = pd.Series(svm_step.coef_[0], index=engineered_features).sort_values()

coef_to_plot = pd.concat([coef.head(6), coef.tail(6)])
plt.figure(figsize=(8, 6))
sns.barplot(x=coef_to_plot.values, y=coef_to_plot.index, hue=coef_to_plot.index, palette='vlag', legend=False)
plt.axvline(0, color='black', linewidth=1)
plt.title('Linear SVM Coefficients')
plt.xlabel('Coefficient')
plt.ylabel('Feature')
plt.show()

coef.sort_values(ascending=False).to_frame('coefficient').round(4)


## 6. Optional: Small Parameter Search

This section does not run by default. Set `RUN_GRID_SEARCH = True` to try different values for `C` and `class_weight`.


In [ ]:
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    svm_grid = GridSearchCV(
        make_pipeline(StandardScaler(), LinearSVC(max_iter=10000, random_state=RANDOM_STATE)),
        param_grid={
            'linearsvc__C': [0.003, 0.01, 0.03, 0.1, 0.3, 1],
            'linearsvc__class_weight': [None, 'balanced'],
        },
        scoring='accuracy',
        cv=3,
        n_jobs=1,
    )
    svm_grid.fit(X_train, y_train)
    best_svm = svm_grid.best_estimator_
    best_svm_pred = best_svm.predict(X_test)
    best_svm_score = best_svm.decision_function(X_test)
    print('Best parameters:', svm_grid.best_params_)
    display(round_metric_table(pd.DataFrame([
        make_metric_row('Linear SVM tuned', y_test, best_svm_pred, best_svm_score)
    ])))
else:
    print('Grid search skipped. Set RUN_GRID_SEARCH = True to run it.')


## 7. Summary

Linear SVM and Logistic Regression perform very similarly here. Both are linear models, so the improvement is limited.

Among these three models, Random Forest is usually slightly better. The gap is small, which suggests that the dataset may have limited predictive signal or may need stronger feature engineering.
